# Bagging with Undersampling: An Ensemble Approach to Imbalanced Medical Imaging

## Learning Objectives

By the end of this notebook, you will understand:

1. **Why traditional undersampling loses information** - and how to quantify this loss
2. **How bagging with undersampling recovers lost information** - through ensemble diversity
3. **How to implement a 10-model CNN ensemble** - with different balanced training sets
4. **How to evaluate and compare** - against baseline approaches

---

## The Clinical Problem

Early detection of Alzheimer's disease is critical for patient care. MRI scans can reveal structural brain changes, but:

- **Healthy patients vastly outnumber patients with dementia** in most datasets
- **Missing a dementia case (false negative) has higher clinical cost** than a false alarm
- **Standard models trained on imbalanced data tend to predict the majority class**

---

## The Technique: Bagging with Undersampling

Instead of one big model, we train **K smaller models** (K=10 in this notebook).

**For each model:**
- Keep **ALL minority class samples** (no information loss!)
- Randomly sample **N majority class samples** (where N = minority count)
- Train on this **perfectly balanced** dataset

**Key insight:** Each model sees a different random subset of the majority class. Collectively, the ensemble "sees" much more of the data than any single undersampled model.

**Final prediction:** Average the probability outputs from all K models (soft voting).

---
# Part 1: Environment Setup

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Data Processing
import numpy as np
import pandas as pd
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML Utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, 
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    matthews_corrcoef, cohen_kappa_score,
    roc_auc_score, roc_curve
)

# Dataset
from datasets import load_dataset

# Utilities
import random
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# REPRODUCIBILITY
# =============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================
device = (
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)
print(f"Using device: {device}")

---
# Part 2: Data Loading & Binary Conversion

In [ ]:
# =============================================================================
# LOAD THE ALZHEIMER'S MRI DATASET
# =============================================================================
# Source: Hugging Face - Falah/Alzheimer_MRI
# Contains grayscale brain MRI scans with 4 cognitive impairment categories

print("Loading Alzheimer's MRI dataset from Hugging Face...")
dataset = load_dataset('Falah/Alzheimer_MRI', split='train')
print(f"Total samples: {len(dataset)}")

# Original 4-class labels
ORIGINAL_LABELS = {
    0: "Mild Demented",
    1: "Moderate Demented", 
    2: "Non Demented",
    3: "Very Mild Demented"
}

# Count original distribution
original_counts = Counter(ex['label'] for ex in dataset)
print("\nOriginal 4-Class Distribution:")
for label, name in ORIGINAL_LABELS.items():
    count = original_counts[label]
    pct = count / len(dataset) * 100
    print(f"  {name:20s}: {count:4d} samples ({pct:5.1f}%)")

In [ ]:
# =============================================================================
# BINARY LABEL CONVERSION
# =============================================================================
# For clearer educational demonstration, we convert to binary classification:
#   - Class 0: Non-Demented (originally class 2)
#   - Class 1: Demented (originally classes 0, 1, 3 - all dementia severities)

def to_binary_label(label):
    """Convert 4-class label to binary: Non-Demented (0) vs Demented (1)."""
    return 0 if label == 2 else 1

BINARY_LABELS = {0: "Non-Demented", 1: "Demented"}

# Convert all labels
binary_labels = [to_binary_label(ex['label']) for ex in dataset]
binary_counts = Counter(binary_labels)

print("Binary Classification Distribution:")
for label, name in BINARY_LABELS.items():
    count = binary_counts[label]
    pct = count / len(dataset) * 100
    print(f"  {name:15s}: {count:4d} samples ({pct:5.1f}%)")

# Calculate imbalance ratio
majority_count = max(binary_counts.values())
minority_count = min(binary_counts.values())
imbalance_ratio = majority_count / minority_count
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")

In [ ]:
# =============================================================================
# VISUALIZE CLASS DISTRIBUTION
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original 4-class distribution
ax1 = axes[0]
names = [ORIGINAL_LABELS[i] for i in range(4)]
counts = [original_counts[i] for i in range(4)]
colors = ['#ff9999', '#ff4444', '#66b3ff', '#ffcc99']
bars1 = ax1.bar(names, counts, color=colors, edgecolor='black')
ax1.set_ylabel('Number of Samples', fontsize=12)
ax1.set_title('Original 4-Class Distribution', fontsize=14)
ax1.tick_params(axis='x', rotation=30)
for bar, count in zip(bars1, counts):
    ax1.annotate(f'{count}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=11)

# Binary distribution
ax2 = axes[1]
names_bin = list(BINARY_LABELS.values())
counts_bin = [binary_counts[i] for i in range(2)]
colors_bin = ['#66b3ff', '#ff6b6b']
bars2 = ax2.bar(names_bin, counts_bin, color=colors_bin, edgecolor='black')
ax2.set_ylabel('Number of Samples', fontsize=12)
ax2.set_title('Binary Classification Distribution', fontsize=14)
for bar, count in zip(bars2, counts_bin):
    ax2.annotate(f'{count}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nNote: After binary conversion, classes are nearly balanced (2053 vs 2043).")
print("This is actually favorable, but the technique remains valuable for educational")
print("purposes and scenarios with more severe imbalance.")

---
# Part 3: The Problem with Simple Undersampling

**Simple undersampling** randomly discards majority class samples until both classes are equal.

**The problem:** We permanently lose information from the discarded samples.

In [ ]:
# =============================================================================
# QUANTIFY INFORMATION LOSS FROM SIMPLE UNDERSAMPLING
# =============================================================================

samples_lost = majority_count - minority_count
pct_lost = (samples_lost / majority_count) * 100

print("=" * 60)
print("THE COST OF SIMPLE UNDERSAMPLING")
print("=" * 60)
print(f"\nMajority class (Non-Demented): {majority_count} samples")
print(f"Minority class (Demented):     {minority_count} samples")
print(f"\nTo balance via undersampling:")
print(f"  Samples to DISCARD: {samples_lost} ({pct_lost:.1f}% of majority class)")
print(f"  Resulting dataset:  {minority_count * 2} samples (from {len(dataset)})")
print(f"\nThis is {samples_lost} brain scans we'll never learn from!")

---
# Part 4: The Solution - Bagging with Undersampling

## Key Insight

Instead of training **one model** on **one undersampled dataset**, train **K models** on **K different undersampled datasets**.

## How It Works

For each model $k$ (where $k = 1, 2, ..., K$):

$$\text{TrainingSet}_k = \text{ALL minority samples} + \text{Random}(N, \text{majority samples})$$

Where $N$ = number of minority samples.

## Ensemble Prediction

Final prediction uses **soft voting** (probability averaging):

$$P(y|x) = \frac{1}{K} \sum_{k=1}^{K} P_k(y|x)$$

## Why K=10?

- **More models = more data coverage** from the majority class
- **Diminishing returns** after ~10 models for most datasets
- **Computational trade-off**: 10x training time vs single model

In [ ]:
# =============================================================================
# VISUALIZE BAGGING vs SIMPLE UNDERSAMPLING
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Simple Undersampling
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.set_aspect('equal')

# Draw majority class (many samples)
np.random.seed(42)
maj_x = np.random.uniform(0.5, 4.5, 30)
maj_y = np.random.uniform(0.5, 9.5, 30)

# Draw minority class (fewer samples) 
min_x = np.random.uniform(5.5, 9.5, 10)
min_y = np.random.uniform(0.5, 9.5, 10)

# Show discarded samples (faded)
ax1.scatter(maj_x[10:], maj_y[10:], c='lightgray', s=100, alpha=0.5, label='Discarded (lost!)')
# Show kept samples
ax1.scatter(maj_x[:10], maj_y[:10], c='#66b3ff', s=100, edgecolor='black', label='Kept Majority')
ax1.scatter(min_x, min_y, c='#ff6b6b', s=100, edgecolor='black', label='All Minority')

ax1.set_title('Simple Undersampling\n(20 samples lost forever)', fontsize=13)
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)
ax1.axis('off')

# Right: Bagging with Undersampling
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_aspect('equal')

# Show all majority samples with different colors for different subsets
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for i in range(3):  # Show 3 different model subsets
    np.random.seed(42 + i)
    subset_idx = np.random.choice(30, 10, replace=False)
    offset = i * 0.15
    ax2.scatter(maj_x[subset_idx] + offset, maj_y[subset_idx] + offset, 
               c=[colors[i]], s=80, alpha=0.7, edgecolor='black',
               label=f'Model {i+1} subset' if i < 3 else None)

# Minority class (always all included)
ax2.scatter(min_x, min_y, c='#ff6b6b', s=100, edgecolor='black', 
           label='All Minority (always)', zorder=10)

ax2.set_title('Bagging with Undersampling\n(Each model sees different majority subset)', fontsize=13)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)
ax2.axis('off')

plt.tight_layout()
plt.savefig('bagging_vs_simple.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 5: Dataset and Model Implementation

In [ ]:
# =============================================================================
# TRAIN/VALIDATION SPLIT
# =============================================================================
# Stratified 80/20 split BEFORE creating ensemble subsets
# This ensures all models are evaluated on the SAME validation set

train_idx, val_idx = train_test_split(
    np.arange(len(dataset)), 
    test_size=0.2, 
    stratify=binary_labels, 
    random_state=SEED
)

print(f"Training set: {len(train_idx)} samples")
print(f"Validation set: {len(val_idx)} samples")

# Analyze training set distribution
train_binary_labels = [to_binary_label(dataset[i]['label']) for i in train_idx]
train_counts = Counter(train_binary_labels)

# Identify minority/majority in training set
minority_class = min(train_counts, key=train_counts.get)
majority_class = 1 - minority_class
n_minority = train_counts[minority_class]
n_majority = train_counts[majority_class]

print(f"\nTraining set distribution:")
print(f"  {BINARY_LABELS[minority_class]} (minority): {n_minority}")
print(f"  {BINARY_LABELS[majority_class]} (majority): {n_majority}")

In [ ]:
# =============================================================================
# PYTORCH DATASET CLASS
# =============================================================================

class BinaryAlzheimersDataset(Dataset):
    """
    PyTorch Dataset for binary Alzheimer's classification.
    
    Converts grayscale MRI images to tensors and applies binary label conversion.
    
    Args:
        hf_dataset: Hugging Face dataset object
        indices: List of indices to include in this subset
        transform: Optional torchvision transforms to apply
    """
    def __init__(self, hf_dataset, indices, transform=None):
        self.data = hf_dataset
        self.indices = list(indices)  # Ensure it's a list
        self.transform = transform
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        # Get the actual dataset index
        real_idx = self.indices[idx]
        
        # Load and normalize image (0-255 -> 0-1)
        img = np.array(self.data[real_idx]['image'], dtype=np.float32) / 255.0
        
        # Add channel dimension: (H, W) -> (1, H, W)
        img = torch.tensor(img).unsqueeze(0)
        
        # Apply transforms if specified
        if self.transform:
            img = self.transform(img)
        
        # Convert to binary label
        label = to_binary_label(self.data[real_idx]['label'])
        
        return img, torch.tensor(label, dtype=torch.long)

print("BinaryAlzheimersDataset class defined.")

In [ ]:
# =============================================================================
# CNN MODEL ARCHITECTURE
# =============================================================================
# Reused from existing codebase (CNN-Pytorch.ipynb)
# Modified: n_classes=2 for binary classification

class OptimizedCNN(nn.Module):
    """
    Configurable CNN with 2-3 convolutional layers, dropout, and batch normalization.
    
    Architecture:
        Conv1 (f1 filters) -> BN -> ReLU -> MaxPool -> Dropout
        Conv2 (f2 filters) -> BN -> ReLU -> MaxPool -> Dropout
        [Conv3 (f3 filters) -> BN -> ReLU -> MaxPool -> Dropout]  (if n_layers >= 3)
        Flatten -> FC1 (fc_size) -> ReLU -> Dropout -> FC2 (n_classes)
    
    Args:
        n_layers: Number of conv layers (2 or 3)
        f1, f2, f3: Filter counts for each conv layer
        dropout: Dropout probability
        fc_size: Size of the fully connected hidden layer
        n_classes: Number of output classes (2 for binary)
    """
    def __init__(self, n_layers=3, f1=32, f2=64, f3=128, dropout=0.3, fc_size=256, n_classes=2):
        super().__init__()
        self.n_layers = n_layers
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, f1, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(f1)
        
        self.conv2 = nn.Conv2d(f1, f2, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(f2)
        
        if n_layers >= 3:
            self.conv3 = nn.Conv2d(f2, f3, 3, padding=1)
            self.bn3 = nn.BatchNorm2d(f3)
        
        # Pooling and dropout
        self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout2d(dropout)
        
        # Calculate flattened size (assuming 128x128 input)
        # After 3 pools: 128 -> 64 -> 32 -> 16
        # After 2 pools: 128 -> 64 -> 32
        flat_size = f3 * 16 * 16 if n_layers >= 3 else f2 * 32 * 32
        
        # Fully connected layers
        self.fc1 = nn.Linear(flat_size, fc_size)
        self.fc2 = nn.Linear(fc_size, n_classes)
        self.drop_fc = nn.Dropout(dropout)

    def forward(self, x):
        # Conv block 1
        x = self.drop(self.pool(F.relu(self.bn1(self.conv1(x)))))
        # Conv block 2
        x = self.drop(self.pool(F.relu(self.bn2(self.conv2(x)))))
        # Conv block 3 (optional)
        if self.n_layers >= 3:
            x = self.drop(self.pool(F.relu(self.bn3(self.conv3(x)))))
        # Fully connected
        x = x.flatten(1)
        x = self.drop_fc(F.relu(self.fc1(x)))
        return self.fc2(x)

# Test model instantiation
test_model = OptimizedCNN(n_classes=2)
print(f"Model parameters: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model

---
# Part 6: Creating Balanced Subsets for Each Model

This is the **core of the bagging technique**:
- Each of our 10 models gets a different training set
- ALL minority samples are included in EVERY set
- A DIFFERENT random sample of majority is used for each

In [ ]:
# =============================================================================
# BALANCED SUBSET CREATION FUNCTION
# =============================================================================

def create_balanced_subset_indices(train_indices, train_labels, minority_class, seed):
    """
    Create indices for a balanced training subset.
    
    This function implements the undersampling portion of "bagging with undersampling":
    - Keeps ALL samples from the minority class
    - Randomly samples an equal number from the majority class
    
    Args:
        train_indices: Array of indices into the full dataset (training portion)
        train_labels: Binary labels corresponding to train_indices
        minority_class: The class label (0 or 1) that is the minority
        seed: Random seed for reproducibility and creating different subsets
    
    Returns:
        List of indices forming a balanced training subset
    """
    np.random.seed(seed)
    
    # Separate indices by class
    minority_indices = [idx for idx, lbl in zip(train_indices, train_labels) 
                        if lbl == minority_class]
    majority_indices = [idx for idx, lbl in zip(train_indices, train_labels) 
                        if lbl != minority_class]
    
    # Sample majority to match minority count (without replacement)
    n_minority = len(minority_indices)
    sampled_majority = np.random.choice(majority_indices, size=n_minority, replace=False)
    
    # Combine and shuffle
    balanced_indices = minority_indices + list(sampled_majority)
    np.random.shuffle(balanced_indices)
    
    return balanced_indices

print("Balanced subset creation function defined.")

In [ ]:
# =============================================================================
# CREATE 10 DIFFERENT BALANCED SUBSETS
# =============================================================================

N_MODELS = 10  # Number of models in our ensemble

# Create a balanced subset for each model (different random seed each time)
subset_indices = [
    create_balanced_subset_indices(train_idx, train_binary_labels, minority_class, SEED + i)
    for i in range(N_MODELS)
]

# Verify each subset is balanced
print(f"Created {N_MODELS} balanced training subsets:\n")
print(f"{'Model':<8} {'Non-Demented':>14} {'Demented':>10} {'Total':>8}")
print("-" * 44)

for i, indices in enumerate(subset_indices):
    labels = [to_binary_label(dataset[idx]['label']) for idx in indices]
    counts = Counter(labels)
    print(f"Model {i+1:<3} {counts[0]:>14} {counts[1]:>10} {len(indices):>8}")

In [ ]:
# =============================================================================
# VISUALIZE SUBSET OVERLAP
# =============================================================================
# How much do the majority class samples overlap between models?

# Extract majority class indices from each subset
majority_per_model = []
for indices in subset_indices:
    maj_in_subset = set(idx for idx in indices 
                        if to_binary_label(dataset[idx]['label']) == majority_class)
    majority_per_model.append(maj_in_subset)

# Compute pairwise overlap
overlap_matrix = np.zeros((N_MODELS, N_MODELS))
for i in range(N_MODELS):
    for j in range(N_MODELS):
        intersection = len(majority_per_model[i] & majority_per_model[j])
        union = len(majority_per_model[i] | majority_per_model[j])
        overlap_matrix[i, j] = intersection / union if union > 0 else 0  # Jaccard similarity

# Plot
plt.figure(figsize=(8, 7))
sns.heatmap(overlap_matrix, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=[f'M{i+1}' for i in range(N_MODELS)],
            yticklabels=[f'M{i+1}' for i in range(N_MODELS)])
plt.title('Majority Class Overlap Between Models\n(Jaccard Similarity)', fontsize=13)
plt.xlabel('Model')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig('subset_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate average overlap
avg_overlap = (overlap_matrix.sum() - N_MODELS) / (N_MODELS * (N_MODELS - 1))
print(f"\nAverage pairwise Jaccard similarity: {avg_overlap:.3f}")
print("Low overlap means models see different portions of majority class -> more diversity!")

---
# Part 7: Training Functions

In [ ]:
# =============================================================================
# DATA AUGMENTATION
# =============================================================================
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
])

print("Data augmentation transforms defined:")
print("  - Random horizontal flip (50%)")
print("  - Random rotation (±15°)")
print("  - Random translation (±10%)")

In [ ]:
# =============================================================================
# TRAINING FUNCTION
# =============================================================================

def train_single_model(model, train_loader, val_loader, epochs=15, lr=1e-3, patience=5):
    """
    Train a single CNN model with early stopping.
    
    Args:
        model: PyTorch model to train
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        epochs: Maximum number of training epochs
        lr: Learning rate
        patience: Early stopping patience (epochs without improvement)
    
    Returns:
        best_state: State dict of best model (by validation accuracy)
        history: Dict with training/validation loss and accuracy per epoch
        best_val_acc: Best validation accuracy achieved
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        # === Training Phase ===
        model.train()
        train_loss, correct, total = 0, 0, 0
        
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            correct += (outputs.argmax(1) == lbls).sum().item()
            total += lbls.size(0)
        
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(correct / total)
        
        # === Validation Phase ===
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                val_loss += criterion(outputs, lbls).item()
                val_correct += (outputs.argmax(1) == lbls).sum().item()
                val_total += lbls.size(0)
        
        val_acc = val_correct / val_total
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(val_acc)
        
        # Learning rate scheduling
        scheduler.step(val_loss / len(val_loader))
        
        # Early stopping check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    return best_state, history, best_val_acc

print("Training function defined.")

---
# Part 8: Train the Ensemble

In [ ]:
# =============================================================================
# HYPERPARAMETERS
# =============================================================================

BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 5

print("Training Configuration:")
print(f"  Number of models: {N_MODELS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs per model: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Early stopping patience: {EARLY_STOPPING_PATIENCE}")

In [ ]:
# =============================================================================
# CREATE VALIDATION DATALOADER (SHARED ACROSS ALL MODELS)
# =============================================================================

val_dataset = BinaryAlzheimersDataset(dataset, val_idx)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    pin_memory=(device == 'cuda'),
    num_workers=0
)

print(f"Validation DataLoader created with {len(val_dataset)} samples.")

In [ ]:
# =============================================================================
# TRAIN ALL ENSEMBLE MODELS
# =============================================================================

ensemble_models = []      # Will store best state_dict for each model
ensemble_histories = []   # Will store training history for each model
individual_accuracies = [] # Will store best validation accuracy for each model

print("=" * 70)
print(f"TRAINING ENSEMBLE OF {N_MODELS} MODELS (Bagging with Undersampling)")
print("=" * 70)

for i in range(N_MODELS):
    print(f"\n[Model {i+1}/{N_MODELS}]")
    
    # Create balanced training set for this model
    train_subset = BinaryAlzheimersDataset(
        dataset, 
        subset_indices[i], 
        transform=train_transforms
    )
    train_loader = DataLoader(
        train_subset, 
        batch_size=BATCH_SIZE, 
        shuffle=True,
        pin_memory=(device == 'cuda'),
        num_workers=0
    )
    
    print(f"  Training samples: {len(train_subset)} (balanced)")
    
    # Create fresh model
    model = OptimizedCNN(
        n_layers=3, f1=32, f2=64, f3=128, 
        dropout=0.3, fc_size=256, n_classes=2
    ).to(device)
    
    # Train
    best_state, history, best_acc = train_single_model(
        model, train_loader, val_loader,
        epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE
    )
    
    # Store results
    ensemble_models.append(best_state)
    ensemble_histories.append(history)
    individual_accuracies.append(best_acc)
    
    print(f"  Best Val Accuracy: {best_acc:.4f}")
    print(f"  Epochs trained: {len(history['train_loss'])}")

print("\n" + "=" * 70)
print("ENSEMBLE TRAINING COMPLETE")
print("=" * 70)
print(f"\nIndividual model accuracies: {[f'{a:.3f}' for a in individual_accuracies]}")
print(f"Mean: {np.mean(individual_accuracies):.4f} | Std: {np.std(individual_accuracies):.4f}")

---
# Part 9: Ensemble Prediction (Soft Voting)

In [ ]:
# =============================================================================
# ENSEMBLE PREDICTION FUNCTION
# =============================================================================

def ensemble_predict(model_states, val_loader, device):
    """
    Get ensemble predictions using soft voting (probability averaging).
    
    For each input sample:
    1. Get probability predictions from all K models
    2. Average the probabilities
    3. Take argmax of averaged probabilities
    
    Args:
        model_states: List of state_dict for each model in ensemble
        val_loader: DataLoader for validation/test data
        device: Device to run inference on
    
    Returns:
        ensemble_preds: Final class predictions (numpy array)
        ensemble_probs: Averaged probabilities (numpy array, shape [n_samples, n_classes])
        all_model_probs: Individual model probabilities (list of arrays)
    """
    all_model_probs = []
    
    for i, state_dict in enumerate(model_states):
        # Load model with saved weights
        model = OptimizedCNN(
            n_layers=3, f1=32, f2=64, f3=128,
            dropout=0.3, fc_size=256, n_classes=2
        ).to(device)
        model.load_state_dict(state_dict)
        model.eval()
        
        # Get probabilities for all validation samples
        model_probs = []
        with torch.no_grad():
            for imgs, _ in val_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                probs = F.softmax(outputs, dim=1)
                model_probs.append(probs.cpu().numpy())
        
        all_model_probs.append(np.vstack(model_probs))
    
    # Average probabilities across all models
    ensemble_probs = np.mean(all_model_probs, axis=0)
    
    # Final predictions from averaged probabilities
    ensemble_preds = ensemble_probs.argmax(axis=1)
    
    return ensemble_preds, ensemble_probs, all_model_probs

print("Ensemble prediction function defined.")

In [ ]:
# =============================================================================
# GET ENSEMBLE PREDICTIONS
# =============================================================================

# Ground truth labels for validation set
val_labels = [to_binary_label(dataset[idx]['label']) for idx in val_idx]

# Get ensemble predictions
print("Running ensemble inference...")
ensemble_preds, ensemble_probs, all_model_probs = ensemble_predict(
    ensemble_models, val_loader, device
)

print(f"\nPredictions shape: {ensemble_preds.shape}")
print(f"Probabilities shape: {ensemble_probs.shape}")

---
# Part 10: Comprehensive Metrics Evaluation

In [ ]:
# =============================================================================
# METRICS COMPUTATION FUNCTION
# =============================================================================

def compute_binary_metrics(y_true, y_pred, y_prob=None):
    """
    Compute comprehensive metrics for binary classification.
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
        y_prob: Prediction probabilities (optional, for ROC-AUC)
    
    Returns:
        Dictionary of metrics
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'kappa': cohen_kappa_score(y_true, y_pred),
    }
    
    if y_prob is not None:
        # Use probability of positive class for ROC-AUC
        prob_positive = y_prob[:, 1] if len(y_prob.shape) > 1 else y_prob
        metrics['roc_auc'] = roc_auc_score(y_true, prob_positive)
    
    return metrics

print("Metrics function defined.")

In [ ]:
# =============================================================================
# COMPUTE AND DISPLAY ENSEMBLE METRICS
# =============================================================================

ensemble_metrics = compute_binary_metrics(val_labels, ensemble_preds, ensemble_probs)

print("=" * 55)
print("BAGGING WITH UNDERSAMPLING ENSEMBLE - RESULTS")
print("=" * 55)
print(f"\n{'Metric':<25} {'Value':>10}")
print("-" * 37)
for metric, value in ensemble_metrics.items():
    print(f"{metric:<25} {value:>10.4f}")

---
# Part 11: Baseline Comparison

To understand the benefit of bagging with undersampling, we compare against:

1. **Baseline**: Single CNN on full unbalanced training data
2. **Simple Undersampling**: Single CNN on one balanced subset
3. **Weighted Loss**: Single CNN with class-weighted CrossEntropyLoss

In [ ]:
# =============================================================================
# HELPER: GET PREDICTIONS FROM A SINGLE MODEL
# =============================================================================

def get_model_predictions(state_dict, val_loader, device):
    """
    Get predictions and probabilities from a single trained model.
    """
    model = OptimizedCNN(
        n_layers=3, f1=32, f2=64, f3=128,
        dropout=0.3, fc_size=256, n_classes=2
    ).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    
    preds, probs = [], []
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds.extend(outputs.argmax(1).cpu().numpy())
            probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
    
    return np.array(preds), np.array(probs)

print("Helper function defined.")

In [ ]:
# =============================================================================
# TRAIN BASELINE: FULL UNBALANCED DATA
# =============================================================================

print("Training Baseline (full unbalanced data)...")

# Create full training dataset (no undersampling)
full_train_dataset = BinaryAlzheimersDataset(
    dataset, list(train_idx), transform=train_transforms
)
full_train_loader = DataLoader(
    full_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=(device == 'cuda'), num_workers=0
)

# Train
model_baseline = OptimizedCNN(n_classes=2).to(device)
baseline_state, baseline_history, baseline_acc = train_single_model(
    model_baseline, full_train_loader, val_loader,
    epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE
)

print(f"  Best Val Accuracy: {baseline_acc:.4f}")

In [ ]:
# =============================================================================
# TRAIN SIMPLE UNDERSAMPLING: ONE BALANCED SUBSET
# =============================================================================

print("Training Simple Undersampling (single balanced subset)...")

# Use the first subset (same as Model 1 of ensemble)
simple_train_dataset = BinaryAlzheimersDataset(
    dataset, subset_indices[0], transform=train_transforms
)
simple_train_loader = DataLoader(
    simple_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=(device == 'cuda'), num_workers=0
)

# Train
model_simple = OptimizedCNN(n_classes=2).to(device)
simple_state, simple_history, simple_acc = train_single_model(
    model_simple, simple_train_loader, val_loader,
    epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE
)

print(f"  Best Val Accuracy: {simple_acc:.4f}")

In [ ]:
# =============================================================================
# TRAIN WEIGHTED LOSS: CLASS-WEIGHTED CROSSENTROPY
# =============================================================================

def train_weighted_model(model, train_loader, val_loader, class_weights, 
                         epochs=15, lr=1e-3, patience=5):
    """
    Train model with weighted CrossEntropyLoss.
    """
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss, correct, total = 0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            correct += (outputs.argmax(1) == lbls).sum().item()
            total += lbls.size(0)
        
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(correct / total)
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                outputs = model(imgs)
                val_loss += criterion(outputs, lbls).item()
                val_correct += (outputs.argmax(1) == lbls).sum().item()
                val_total += lbls.size(0)
        
        val_acc = val_correct / val_total
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(val_acc)
        scheduler.step(val_loss / len(val_loader))
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    return best_state, history, best_val_acc

print("Training Weighted Loss (class-weighted CrossEntropy)...")

# Compute class weights (inverse frequency)
class_weights = torch.tensor([
    len(train_binary_labels) / (2 * train_counts[0]),
    len(train_binary_labels) / (2 * train_counts[1])
], dtype=torch.float32)

print(f"  Class weights: {class_weights.numpy()}")

# Train
model_weighted = OptimizedCNN(n_classes=2).to(device)
weighted_state, weighted_history, weighted_acc = train_weighted_model(
    model_weighted, full_train_loader, val_loader, class_weights,
    epochs=EPOCHS, lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE
)

print(f"  Best Val Accuracy: {weighted_acc:.4f}")

In [ ]:
# =============================================================================
# GET PREDICTIONS FOR ALL BASELINES
# =============================================================================

baseline_preds, baseline_probs = get_model_predictions(baseline_state, val_loader, device)
simple_preds, simple_probs = get_model_predictions(simple_state, val_loader, device)
weighted_preds, weighted_probs = get_model_predictions(weighted_state, val_loader, device)

print("Predictions collected for all approaches.")

---
# Part 12: Results Comparison

In [ ]:
# =============================================================================
# COMPUTE METRICS FOR ALL APPROACHES
# =============================================================================

comparison = {
    '1. Baseline (Unbalanced)': compute_binary_metrics(val_labels, baseline_preds, baseline_probs),
    '2. Simple Undersampling': compute_binary_metrics(val_labels, simple_preds, simple_probs),
    '3. Weighted Loss': compute_binary_metrics(val_labels, weighted_preds, weighted_probs),
    '4. Bagging Ensemble (Ours)': ensemble_metrics,
}

# Create comparison DataFrame
df_comparison = pd.DataFrame(comparison).T

print("=" * 90)
print("COMPREHENSIVE RESULTS COMPARISON")
print("=" * 90)
print(df_comparison.round(4).to_string())

In [ ]:
# =============================================================================
# SAVE COMPARISON RESULTS
# =============================================================================

df_comparison.to_csv('bagging_comparison_results.csv')
print("\nResults saved to bagging_comparison_results.csv")

In [ ]:
# =============================================================================
# VISUALIZE COMPARISON: GROUPED BAR CHART
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 7))

metrics_to_plot = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'mcc', 'roc_auc']
x = np.arange(len(metrics_to_plot))
width = 0.2

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
approaches = list(comparison.keys())

for i, (approach, metrics) in enumerate(comparison.items()):
    values = [metrics.get(m, 0) for m in metrics_to_plot]
    bars = ax.bar(x + i * width, values, width, label=approach, color=colors[i], edgecolor='black')

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Performance Comparison: Bagging Ensemble vs Baselines', fontsize=14)
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([m.replace('_', '\n') for m in metrics_to_plot])
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
ax.set_ylim(0, 1.1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('bagging_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 13: Visualization Suite

In [ ]:
# =============================================================================
# PLOT 1: TRAINING CURVES FOR ALL ENSEMBLE MODELS
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 1, N_MODELS))

# Loss curves
ax1 = axes[0]
for i, history in enumerate(ensemble_histories):
    epochs_range = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs_range, history['train_loss'], color=colors[i], alpha=0.7, 
             label=f'Model {i+1}' if i < 5 else None)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Training Loss', fontsize=12)
ax1.set_title('Training Loss (All Ensemble Models)', fontsize=13)
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2 = axes[1]
for i, history in enumerate(ensemble_histories):
    epochs_range = range(1, len(history['val_acc']) + 1)
    ax2.plot(epochs_range, history['val_acc'], color=colors[i], alpha=0.7)

# Add mean line
ax2.axhline(y=np.mean(individual_accuracies), color='black', linestyle='--', 
            linewidth=2, label=f'Mean ({np.mean(individual_accuracies):.3f})')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Validation Accuracy', fontsize=12)
ax2.set_title('Validation Accuracy (All Ensemble Models)', fontsize=13)
ax2.legend(loc='lower right', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bagging_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# PLOT 2: CONFUSION MATRICES (2x2 GRID)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

approaches_cm = [
    (baseline_preds, '1. Baseline (Unbalanced)'),
    (simple_preds, '2. Simple Undersampling'),
    (weighted_preds, '3. Weighted Loss'),
    (ensemble_preds, '4. Bagging Ensemble (Ours)')
]

for ax, (preds, name) in zip(axes.flat, approaches_cm):
    cm = confusion_matrix(val_labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=list(BINARY_LABELS.values()))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name, fontsize=12)

plt.suptitle('Confusion Matrix Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('bagging_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# PLOT 3: ROC CURVES COMPARISON
# =============================================================================

fig, ax = plt.subplots(figsize=(9, 8))

approaches_roc = [
    (baseline_probs, '1. Baseline', '#636EFA'),
    (simple_probs, '2. Simple Undersampling', '#EF553B'),
    (weighted_probs, '3. Weighted Loss', '#00CC96'),
    (ensemble_probs, '4. Bagging Ensemble', '#AB63FA')
]

for probs, name, color in approaches_roc:
    fpr, tpr, _ = roc_curve(val_labels, probs[:, 1])
    auc = roc_auc_score(val_labels, probs[:, 1])
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

# Random classifier line
ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)', alpha=0.5)

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bagging_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# PLOT 4: MODEL AGREEMENT HEATMAP
# =============================================================================
# How often do individual ensemble models agree with each other?

# Get individual model predictions
individual_preds = []
for state_dict in ensemble_models:
    preds, _ = get_model_predictions(state_dict, val_loader, device)
    individual_preds.append(preds)

individual_preds = np.array(individual_preds)

# Compute agreement matrix
agreement_matrix = np.zeros((N_MODELS, N_MODELS))
for i in range(N_MODELS):
    for j in range(N_MODELS):
        agreement_matrix[i, j] = (individual_preds[i] == individual_preds[j]).mean()

# Plot
plt.figure(figsize=(9, 8))
sns.heatmap(agreement_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.7, vmax=1.0,
            xticklabels=[f'M{i+1}' for i in range(N_MODELS)],
            yticklabels=[f'M{i+1}' for i in range(N_MODELS)])
plt.title('Pairwise Model Agreement (Prediction Overlap)', fontsize=13)
plt.xlabel('Model', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.tight_layout()
plt.savefig('bagging_model_agreement.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistics
avg_agreement = (agreement_matrix.sum() - N_MODELS) / (N_MODELS * (N_MODELS - 1))
print(f"Average pairwise agreement: {avg_agreement:.3f}")
print(f"\nInterpretation: Lower agreement means more model diversity.")
print(f"Diversity is good - it means ensemble members make different errors!")

In [ ]:
# =============================================================================
# PLOT 5: ENSEMBLE CONFIDENCE DISTRIBUTION
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Get max probability (confidence) for each prediction
confidences = ensemble_probs.max(axis=1)
correct_mask = ensemble_preds == np.array(val_labels)

# Plot histograms
ax.hist(confidences[correct_mask], bins=30, alpha=0.7, 
        label=f'Correct ({correct_mask.sum()})', color='green', edgecolor='black')
ax.hist(confidences[~correct_mask], bins=30, alpha=0.7, 
        label=f'Incorrect ({(~correct_mask).sum()})', color='red', edgecolor='black')

ax.axvline(x=0.5, color='black', linestyle='--', label='Decision boundary')
ax.set_xlabel('Prediction Confidence (Max Probability)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Ensemble Confidence Distribution', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bagging_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean confidence on correct predictions: {confidences[correct_mask].mean():.3f}")
print(f"Mean confidence on incorrect predictions: {confidences[~correct_mask].mean():.3f}")

In [ ]:
# =============================================================================
# PLOT 6: RADAR CHART (METRICS COMPARISON)
# =============================================================================

metrics_radar = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'mcc']

# Set up radar chart
angles = np.linspace(0, 2 * np.pi, len(metrics_radar), endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

colors_radar = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

for (name, metrics), color in zip(comparison.items(), colors_radar):
    values = [metrics[m] for m in metrics_radar]
    values += values[:1]  # Close the polygon
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('_', '\n') for m in metrics_radar], fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Metrics Radar Chart', fontsize=14, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)

plt.tight_layout()
plt.savefig('bagging_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# PLOT 7: SAMPLE PREDICTIONS VISUALIZATION
# =============================================================================

def visualize_sample_predictions(val_idx, preds, probs, true_labels, dataset, n_samples=12):
    """
    Visualize sample predictions with confidence scores.
    Shows a mix of correct and incorrect predictions.
    """
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    
    # Identify correct and incorrect predictions
    correct_idx = np.where(preds == np.array(true_labels))[0]
    incorrect_idx = np.where(preds != np.array(true_labels))[0]
    
    # Sample indices (8 correct, up to 4 incorrect)
    np.random.seed(42)
    sample_correct = np.random.choice(correct_idx, min(8, len(correct_idx)), replace=False)
    sample_incorrect = np.random.choice(incorrect_idx, min(4, len(incorrect_idx)), replace=False) \
                       if len(incorrect_idx) > 0 else []
    
    sample_indices = list(sample_correct) + list(sample_incorrect)
    
    for ax, idx in zip(axes.flat, sample_indices):
        real_idx = val_idx[idx]
        img = np.array(dataset[real_idx]['image'])
        
        pred = preds[idx]
        true = true_labels[idx]
        conf = probs[idx].max()
        
        ax.imshow(img, cmap='gray')
        
        is_correct = pred == true
        color = 'green' if is_correct else 'red'
        status = '✓' if is_correct else '✗'
        
        ax.set_title(f'{status} Pred: {BINARY_LABELS[pred]} ({conf:.2f})\n'
                     f'True: {BINARY_LABELS[true]}', 
                     color=color, fontsize=10)
        ax.axis('off')
    
    # Handle any remaining axes (if n_samples < 12)
    for ax in axes.flat[len(sample_indices):]:
        ax.axis('off')
    
    plt.suptitle('Sample Predictions (Green=Correct, Red=Incorrect)', fontsize=14)
    plt.tight_layout()
    return fig

fig = visualize_sample_predictions(val_idx, ensemble_preds, ensemble_probs, val_labels, dataset)
plt.savefig('bagging_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part 14: Statistical Analysis

In [ ]:
# =============================================================================
# VARIANCE REDUCTION ANALYSIS
# =============================================================================

# Per-sample agreement across ensemble models
agreement_per_sample = np.zeros(len(val_labels))
for i in range(len(val_labels)):
    # Count how many models agree with the ensemble prediction
    votes = [individual_preds[k][i] for k in range(N_MODELS)]
    agreement_per_sample[i] = max(Counter(votes).values()) / N_MODELS

print("=" * 60)
print("ENSEMBLE AGREEMENT ANALYSIS")
print("=" * 60)
print(f"\nMean per-sample agreement: {agreement_per_sample.mean():.3f}")
print(f"Min agreement: {agreement_per_sample.min():.3f}")
print(f"Samples with unanimous agreement (100%): {(agreement_per_sample == 1.0).sum()}/{len(val_labels)}")
print(f"Samples with majority vote (>50%): {(agreement_per_sample > 0.5).sum()}/{len(val_labels)}")

In [ ]:
# =============================================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================

def bootstrap_metric(y_true, y_pred, metric_func, n_bootstrap=1000, ci=95):
    """
    Compute bootstrap confidence interval for a metric.
    """
    n = len(y_true)
    scores = []
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        score = metric_func(np.array(y_true)[idx], np.array(y_pred)[idx])
        scores.append(score)
    
    lower = np.percentile(scores, (100 - ci) / 2)
    upper = np.percentile(scores, 100 - (100 - ci) / 2)
    mean = np.mean(scores)
    
    return mean, lower, upper

print("\n" + "=" * 60)
print("BOOTSTRAP CONFIDENCE INTERVALS (95%)")
print("=" * 60)

metrics_to_bootstrap = [
    ('Accuracy', accuracy_score),
    ('Balanced Accuracy', balanced_accuracy_score),
    ('F1 Score', f1_score)
]

print(f"\n{'Metric':<20} {'Mean':>8} {'95% CI':>20}")
print("-" * 50)

for name, func in metrics_to_bootstrap:
    mean, lower, upper = bootstrap_metric(val_labels, ensemble_preds, func)
    print(f"{name:<20} {mean:>8.4f} [{lower:.4f}, {upper:.4f}]")

In [ ]:
# =============================================================================
# ENSEMBLE vs INDIVIDUAL MODEL COMPARISON
# =============================================================================

print("\n" + "=" * 60)
print("ENSEMBLE vs INDIVIDUAL MODELS")
print("=" * 60)

# Compute accuracy for each individual model
individual_accs = []
for i in range(N_MODELS):
    acc = accuracy_score(val_labels, individual_preds[i])
    individual_accs.append(acc)

ensemble_acc = accuracy_score(val_labels, ensemble_preds)

print(f"\nIndividual model accuracies:")
for i, acc in enumerate(individual_accs):
    print(f"  Model {i+1}: {acc:.4f}")

print(f"\nStatistics:")
print(f"  Mean individual: {np.mean(individual_accs):.4f}")
print(f"  Std individual:  {np.std(individual_accs):.4f}")
print(f"  Best individual: {np.max(individual_accs):.4f}")
print(f"  Ensemble:        {ensemble_acc:.4f}")
print(f"\n  Ensemble improvement over mean: +{(ensemble_acc - np.mean(individual_accs))*100:.2f}%")
print(f"  Ensemble improvement over best: +{(ensemble_acc - np.max(individual_accs))*100:.2f}%")

---
# Part 15: Key Observations and Conclusions

## Observation 1: Ensemble vs Individual Models

The ensemble typically outperforms any single constituent model. This is because:
- Each model makes different errors (due to training on different data subsets)
- Averaging predictions smooths out individual errors
- **The whole is greater than the sum of its parts**

## Observation 2: Bagging vs Simple Undersampling

Simple undersampling trains ONE model on ONE balanced subset, permanently discarding majority class samples. Bagging with undersampling:
- Uses ALL minority samples in EVERY model (no minority information loss)
- Uses DIFFERENT majority samples for each model (collectively covers more data)
- **Recovers information that simple undersampling discards**

## Observation 3: When Bagging Helps Most

Bagging with undersampling is most beneficial when:
- Class imbalance is severe (more majority samples to diversify across)
- The majority class has diverse patterns (different subsets capture different patterns)
- Computational resources allow training multiple models

## Observation 4: Computational Trade-off

| Aspect | Single Model | Bagging Ensemble (K=10) |
|--------|--------------|-------------------------|
| Training time | 1x | 10x |
| Inference time | 1x | 10x |
| Memory (training) | 1x | 1x (sequential) |
| Memory (inference) | 1x | 10x (or 1x sequential) |

**Mitigation strategies:**
- Train models in parallel on multiple GPUs
- Use model distillation to compress ensemble into single model
- Use early stopping to reduce per-model training time

## Observation 5: Model Diversity is Key

Ensemble performance depends on **diversity** - models should make different errors. This is achieved by:
- Training on different data subsets (the "bagging" part)
- Random weight initialization
- Data augmentation randomness

Low pairwise agreement between models indicates high diversity, which is desirable!

---
# Part 16: Summary Table

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("=" * 70)
print("FINAL SUMMARY: BAGGING WITH UNDERSAMPLING")
print("=" * 70)

print(f"""
TECHNIQUE:
  - Trained {N_MODELS} CNN models, each on a different balanced subset
  - Each subset: ALL {n_minority} minority samples + random {n_minority} majority samples
  - Final prediction: soft voting (probability averaging)

RESULTS:
  - Ensemble Accuracy:          {ensemble_metrics['accuracy']:.4f}
  - Ensemble Balanced Accuracy: {ensemble_metrics['balanced_accuracy']:.4f}
  - Ensemble F1 Score:          {ensemble_metrics['f1']:.4f}
  - Ensemble ROC-AUC:           {ensemble_metrics['roc_auc']:.4f}

KEY TAKEAWAYS:
  1. Bagging preserves ALL minority class information
  2. Ensemble diversity leads to better generalization
  3. Soft voting (probability averaging) is better than hard voting
  4. Trade-off: 10x computational cost for improved robustness
""")

---
# Part 17: Save the Ensemble

In [ ]:
# =============================================================================
# SAVE ENSEMBLE MODEL
# =============================================================================
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save ensemble
save_dict = {
    'ensemble_models': ensemble_models,
    'config': {
        'n_models': N_MODELS,
        'n_layers': 3,
        'f1': 32, 'f2': 64, 'f3': 128,
        'dropout': 0.3,
        'fc_size': 256,
        'n_classes': 2
    },
    'metrics': ensemble_metrics,
    'individual_accuracies': individual_accuracies,
    'binary_labels': BINARY_LABELS
}

torch.save(save_dict, 'models/bagging_ensemble.pt')
print("Ensemble saved to models/bagging_ensemble.pt")
print(f"File size: {os.path.getsize('models/bagging_ensemble.pt') / 1e6:.2f} MB")

In [ ]:
# =============================================================================
# EXAMPLE: LOADING AND USING THE SAVED ENSEMBLE
# =============================================================================

def load_and_predict(model_path, images, device='cpu'):
    """
    Load saved ensemble and make predictions.
    
    Args:
        model_path: Path to saved .pt file
        images: Tensor of shape (N, 1, H, W)
        device: Device to run on
    
    Returns:
        predictions: Class predictions
        probabilities: Class probabilities
    """
    # Load saved ensemble
    checkpoint = torch.load(model_path, map_location=device)
    config = checkpoint['config']
    
    all_probs = []
    
    for state_dict in checkpoint['ensemble_models']:
        model = OptimizedCNN(
            n_layers=config['n_layers'],
            f1=config['f1'], f2=config['f2'], f3=config['f3'],
            dropout=config['dropout'],
            fc_size=config['fc_size'],
            n_classes=config['n_classes']
        ).to(device)
        model.load_state_dict(state_dict)
        model.eval()
        
        with torch.no_grad():
            probs = F.softmax(model(images.to(device)), dim=1)
            all_probs.append(probs.cpu().numpy())
    
    ensemble_probs = np.mean(all_probs, axis=0)
    ensemble_preds = ensemble_probs.argmax(axis=1)
    
    return ensemble_preds, ensemble_probs

print("Example function for loading and using the ensemble defined.")
print("\nUsage:")
print("  preds, probs = load_and_predict('models/bagging_ensemble.pt', images)")

---
# Part 18: Extensions and Future Work

## Potential Improvements

1. **Heterogeneous Ensemble**: Use different architectures (ResNet, EfficientNet) instead of identical CNNs

2. **Weighted Voting**: Weight each model's vote by its validation performance

3. **SMOTE Integration**: Combine undersampling with synthetic minority oversampling

4. **Feature Bagging**: Instead of sample-space bagging, train on different feature subsets

5. **Multi-Class Extension**: Apply to original 4-class problem with multi-level undersampling

6. **Knowledge Distillation**: Compress ensemble into a single efficient model

## Try It Yourself!

- Change `N_MODELS` to see how ensemble size affects performance
- Modify `create_balanced_subset_indices` to try different sampling strategies
- Compare soft voting vs hard voting
- Add more data augmentation to increase model diversity

In [ ]:
print("\n" + "=" * 70)
print("NOTEBOOK COMPLETE")
print("=" * 70)
print("\nFiles created:")
print("  - class_distribution.png")
print("  - bagging_vs_simple.png")
print("  - subset_overlap.png")
print("  - bagging_training_curves.png")
print("  - bagging_confusion_matrices.png")
print("  - bagging_roc_curves.png")
print("  - bagging_model_agreement.png")
print("  - bagging_confidence_distribution.png")
print("  - bagging_radar_chart.png")
print("  - bagging_sample_predictions.png")
print("  - bagging_comparison_results.csv")
print("  - bagging_comparison_metrics.png")
print("  - models/bagging_ensemble.pt")